In [9]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.metrics import mean_absolute_error
import plotly.graph_objects as go
import os
import json
import datetime as dt
from pathlib import Path
import pandas as pd

In [10]:
def load_sid_split(in_json):
    """
    Load train/test subject IDs from a JSON.
    """
    with open(in_json, "r", encoding="utf-8") as f:
        obj = json.load(f)
    train = set(obj["train_sids"])
    test  = set(obj["test_sids"])
    if train & test:
        raise ValueError("Loaded split has overlapping sids.")
    return train, test, obj.get("meta", {})

# --- APPLY TO ANY DATAFRAME ---

def apply_sid_split(data, train_sids, test_sids, sid_col="sid", sex_col="sex"):
    """
    Given a DataFrame and saved subject IDs, return aligned splits for all/male/female.
    """
    sid_as_str = data[sid_col].astype(str)
    is_train = sid_as_str.isin(train_sids)
    is_test  = sid_as_str.isin(test_sids)

    train_all = data[is_train].copy()
    test_all  = data[is_test].copy()

    male   = data[data[sex_col] == "M"]
    female = data[data[sex_col] == "F"]

    train_m = male[male[sid_col].astype(str).isin(train_sids)].copy()
    test_m  = male[male[sid_col].astype(str).isin(test_sids)].copy()
    train_f = female[female[sid_col].astype(str).isin(train_sids)].copy()
    test_f  = female[female[sid_col].astype(str).isin(test_sids)].copy()

    return {"all": (train_all, test_all),
            "male": (train_m, test_m),
            "female": (train_f, test_f)}

# Train code

In [11]:
def build_nextx_dataset(
    df_long: pd.DataFrame,
    sid_col="sid", landmark_col="landmark",
    age_col="age", x_col="x",
    use_dt=True
):
    """
    Returns:
        X (ndarray): features [x_t,(dt)] per row
        y (ndarray): targets x_{t+1}
        groups (ndarray): group ids for GroupKFold (by sid)
        meta (DataFrame): rows with sid, landmark, age_t, age_tp1, x_t, x_tp1
    """
    cols_needed = {sid_col, landmark_col, age_col, x_col}
    missing = cols_needed - set(df_long.columns)
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    d = df_long[[sid_col, landmark_col, age_col, x_col]].dropna().copy()
    d[age_col] = pd.to_numeric(d[age_col], errors="coerce")
    d[x_col] = pd.to_numeric(d[x_col], errors="coerce")
    d = d.dropna(subset=[age_col, x_col])

    rows = []
    for (sid, lmk), g in d.groupby([sid_col, landmark_col], sort=False):
        g = g.sort_values(age_col)
        xs = g[x_col].to_numpy()
        ages = g[age_col].to_numpy()
        if len(xs) < 2:
            continue
        x_t   = xs[:-1]
        x_tp1 = xs[1:]
        dt    = np.diff(ages)
        # Keep only positive/nonzero steps
        mask = np.isfinite(x_t) & np.isfinite(x_tp1) & np.isfinite(dt) & (dt > 0)
        if not np.any(mask):
            continue
        for xt, xt1, dti, a_t, a_tp1 in zip(x_t[mask], x_tp1[mask], dt[mask], ages[:-1][mask], ages[1:][mask]):
            feats = [xt] + ([dti] if use_dt else [])
            rows.append((sid, lmk, a_t, a_tp1, xt, xt1, *feats))

    if not rows:
        raise ValueError("No usable (x_t → x_{t+1}) pairs found. Check your data.")

    # Assemble
    cols = [sid_col, landmark_col, f"{age_col}_t", f"{age_col}_tp1", "x_t", "x_tp1", "feat_x_t"] + (["feat_dt"] if use_dt else [])
    meta = pd.DataFrame(rows, columns=cols)
    y = meta["x_tp1"].to_numpy().astype(float)
    if use_dt:
        X = meta[["feat_x_t", "feat_dt"]].to_numpy(dtype=float)
    else:
        X = meta[["feat_x_t"]].to_numpy(dtype=float)
    groups = meta[sid_col].astype(str).to_numpy()
    return X, y, groups, meta

# ----------------------------
# 2) Train a simple model (Ridge). Swap for any regressor.
# ----------------------------
def fit_nextx_regressor(df_long, use_dt=True, alpha=1.0, cv_splits=5,x_col='x'):
    X, y, groups, meta = build_nextx_dataset(df_long,x_col=x_col, use_dt=use_dt)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=alpha, random_state=0))
    ])
    # Subject-wise CV
    gkf = GroupKFold(n_splits=min(cv_splits, len(np.unique(groups))))
    scores = cross_val_score(pipe, X, y, groups=groups, cv=gkf, scoring="neg_mean_absolute_error")
    pipe.fit(X, y)
    return pipe, {"cv_MAE(mean)": -scores.mean(), "cv_MAE(std)": scores.std(), "n_pairs": len(y)}, meta

# ----------------------------
# 3) Predict the next x from a current x (and dt)
# ----------------------------
def predict_next_x(model, x_t, dt=None):
    if dt is None:
        X = np.array([[x_t]], float)
    else:
        X = np.array([[x_t, dt]], float)
    return float(model.predict(X)[0])

In [12]:
landmarks = ['sella','nasion','porion','orbitale','u i apex','point a','u i edge','l i edge','point b','l i apex','pogonion','menton','u 6 apex','u 6 cusp','l 6 cusp','l 6 apex','gonion l','gonion u','condyle','pns','basion','u_6_mcp','l_6_mcp','ans','articular','mid gonion']

## get Data

In [13]:
data = pd.read_csv("/data/all_landmark_series_long.csv")
train_sids, test_sids, meta = load_sid_split("/data/splits/sid_split_v1.json")
re_splits = apply_sid_split(data, train_sids, test_sids)

train_all, test_all = re_splits["all"]
train_m, test_m     = re_splits["male"]
train_f, test_f     = re_splits["female"]

## Eval

In [14]:
def _build_nextstep_table(df, axis_col="x", age_col="age", sid_col="sid"):
    """
    From a long df (one landmark already filtered), build next-step (t -> t+1) rows.
    Returns list of dicts: {sid, age_next, val_t, dt, y_true}
    """
    rows = []
    for sid, g in df.groupby(sid_col):
        g = g[[age_col, axis_col]].dropna().sort_values(age_col)
        if len(g) < 2:
            continue
        ages = g[age_col].to_numpy(float)
        vals = g[axis_col].to_numpy(float)
        dts  = np.diff(ages)
        for i in range(len(vals) - 1):
            rows.append({
                "sid": sid,
                "age_next": ages[i+1],
                "val_t": vals[i],
                "dt": dts[i],
                "y_true": vals[i+1],
            })
    return pd.DataFrame(rows)

def _predict_nextstep(model, table, use_dt=True):
    """
    Predict y_{t+1} from val_t (+ dt if use_dt).
    Tries to match model's expected #features (1 or 2).
    """
    if table.empty:
        table["y_pred"] = []
        return table

    # Try to detect feature count; fall back to requested use_dt
    nfeat = getattr(model, "n_features_in_", 2 if use_dt else 1)
    if nfeat == 1:
        X = table[["val_t"]].to_numpy(float)
    else:
        # default to 2 features (val_t, dt) if available
        if use_dt and "dt" in table.columns:
            X = table[["val_t", "dt"]].to_numpy(float)
        else:
            # degrade gracefully if dt isn't used/available
            X = table[["val_t"]].to_numpy(float)

    y_pred = np.asarray(model.predict(X)).reshape(-1)
    out = table.copy()
    out["y_pred"] = y_pred
    return out

def eval_nextxy_test(models_xy, test_df, landmark, use_dt=True):
    """
    Evaluate next-step MAE for x and y on the test split for a given landmark.
    Returns metrics dict and per-axis prediction frames.
    """
    m_x, m_y = models_xy
    # Filter this landmark
    dL = test_df.loc[test_df["landmark"] == landmark].copy()
    if dL.empty:
        return {"test_MAE_x": np.nan, "test_MAE_y": np.nan, "test_MAE_xy": np.nan}, None, None

    # Build next-step tables
    t_x = _build_nextstep_table(dL, axis_col="x")
    t_y = _build_nextstep_table(dL, axis_col="y")

    # Predict
    px = _predict_nextstep(m_x, t_x, use_dt=use_dt) if m_x is not None and not t_x.empty else pd.DataFrame()
    py = _predict_nextstep(m_y, t_y, use_dt=use_dt) if m_y is not None and not t_y.empty else pd.DataFrame()

    mae_x = mean_absolute_error(px["y_true"], px["y_pred"]) if not px.empty else np.nan
    mae_y = mean_absolute_error(py["y_true"], py["y_pred"]) if not py.empty else np.nan
    mae_xy = np.nanmean([mae_x, mae_y])

    return {"test_MAE_x": float(mae_x), "test_MAE_y": float(mae_y), "test_MAE_xy": float(mae_xy)}, px, py

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

def _build_nextstep_1d(df, axis_col="x", age_col="age", sid_col="sid", use_dt=True):
    """
    Returns columns: sid, age_tp1, val_t, dt, y_true
    (y_true is the next value along the chosen axis)
    """
    rows = []
    for sid, g in df.groupby(sid_col):
        g = g[[age_col, axis_col]].dropna().sort_values(age_col)
        if len(g) < 2:
            continue
        ages = g[age_col].to_numpy(float)
        vals = g[axis_col].to_numpy(float)
        dts  = np.diff(ages) if use_dt else np.full(len(ages)-1, np.nan, float)
        for i in range(len(vals) - 1):
            rows.append({
                "sid": sid,
                "age_tp1": ages[i+1],
                "val_t":   vals[i],
                "dt":      float(dts[i]) if use_dt else None,
                "y_true":  vals[i+1],
            })
    return pd.DataFrame(rows)

def _predict_next_from_table(model, t, use_dt=True):
    """Vectorized prediction with graceful fallback to predict_next_x(...)."""
    if t.empty:
        return t
    # Choose feature shape based on model / availability
    nfeat = getattr(model, "n_features_in_", 2 if (use_dt and "dt" in t.columns) else 1)
    if nfeat >= 2 and use_dt and "dt" in t.columns:
        X = t[["val_t", "dt"]].to_numpy(float)
    else:
        X = t[["val_t"]].to_numpy(float)
    try:
        y_pred = np.asarray(model.predict(X)).reshape(-1)
    except Exception:
        # fallback: your helper, works for x or y models
        y_pred = np.asarray([
            predict_next_x(model, vt, dti if (use_dt and "dt" in t.columns) else None)
            for vt, dti in zip(t["val_t"], t.get("dt", [None]*len(t)))
        ])
    t = t.copy()
    t["y_pred"] = y_pred
    return t

# --- Generic plot: Age vs chosen axis (X or Y) ---
def plot_all_subjects_landmark_nextaxis(
    df_landmark, model, axis="x", use_dt=True,
    sid_col="sid", age_col="age", x_col="x", y_col="y",
    title="", out_html=None, tick_mm=4.0
):
    """
    ALL sids overlay:
      • Actual (age -> X or Y): blue lines
      • Predicted next-step (at age_{t+1}): red lines + markers (connected per sid, semi-transparent lines)
      • Dotted vertical line centered on each predicted value
      • Legend on the right with single entries
    Returns Plotly Figure or None (skips quietly if not enough data).
    """
    if df_landmark is None or df_landmark.empty:
        return None

    axis = axis.lower()
    axis_col = x_col if axis == "x" else y_col
    axis_label = "X" if axis == "x" else "Y"

    # 1) Build next-step table for the chosen axis
    t = _build_nextstep_1d(
        df_landmark[[sid_col, age_col, axis_col]],
        axis_col=axis_col, age_col=age_col, sid_col=sid_col, use_dt=use_dt
    )
    if t.empty:
        return None

    # 2) Predict next-step values and compute error vs truth
    t = _predict_next_from_table(model, t, use_dt=use_dt)
    if t.empty:
        return None
    t["abs_err"] = np.abs(t["y_pred"] - t["y_true"])

    # 3) Figure
    fig = go.Figure()

    # Only show one legend item per type
    first_actual = True
    first_pred   = True

    # Actual per-sid (blue)
    for sid, g in df_landmark.groupby(sid_col):
        g = g[[age_col, axis_col]].dropna().sort_values(age_col)
        if len(g) < 2:
            continue
        fig.add_trace(go.Scatter(
            x=g[age_col], y=g[axis_col],
            mode="lines+markers",
            line=dict(color="rgba(0, 102, 255, 1.0)", width=2),
            marker=dict(size=4, color="rgba(0, 102, 255, 1.0)"),
            name="Actual",
            legendgroup="actual",
            showlegend=first_actual,
            hovertemplate=f"sid={sid}<br>age=%{{x:.3f}}<br>{axis_label.lower()}=%{{y:.3f}}<extra></extra>",
        ))
        first_actual = False

    # Predicted per-sid (red, connected; semi-transparent line)
    for sid, g in t.groupby("sid"):
        custom = np.stack([g["y_true"].values, g["abs_err"].values], axis=-1)
        fig.add_trace(go.Scatter(
            x=g["age_tp1"], y=g["y_pred"],
            mode="lines+markers",
            line=dict(color="rgba(220, 0, 0, 0.45)", width=2),  # semi-transparent connected line
            marker=dict(color="red", size=6),                    # solid red markers
            name="Predicted",
            legendgroup="pred",
            showlegend=first_pred,
            customdata=custom,
            hovertemplate=(
                f"sid={sid}<br>age=%{{x:.3f}}"
                f"<br>pred {axis_label}=%{{y:.3f}}"
                f"<br>true {axis_label}=%{{customdata[0]:.3f}}"
                f"<br>|err|=%{{customdata[1]:.3f}}<extra></extra>"
            ),
        ))
        first_pred = False

    # Dotted vertical line centered on each predicted value
    for ax, py in zip(t["age_tp1"], t["y_pred"]):
        fig.add_shape(
            type="line", xref="x", yref="y",
            x0=ax, x1=ax, y0=py - tick_mm/2, y1=py + tick_mm/2,
            line=dict(width=2, dash="dot", color="black"),
            layer="above"
        )

    fig.update_layout(
        title=title or f"Next-step {axis_label} predictions (all sids)",
        xaxis_title="Age",
        yaxis_title=f"{axis_label} (mm)",
        template="plotly_white",
        height=480,
        legend=dict(
            orientation="v",
            x=1.02, xanchor="left",
            y=1.0,  yanchor="top",
            bgcolor="rgba(255,255,255,0.6)",
            bordercolor="lightgray",
            borderwidth=1
        ),
        margin=dict(r=140)  # leave room on the right for legend
    )
    if out_html:
        Path(out_html).parent.mkdir(parents=True, exist_ok=True)
        fig.write_html(out_html)
    return fig



# --- Thin wrappers to match your old calls ---
def plot_all_subjects_landmark_nextx(
    df_landmark, model_x, use_dt=True,
    sid_col="sid", age_col="age", x_col="x",
    title="", out_html=None, tick_mm=4.0
):
    return plot_all_subjects_landmark_nextaxis(
        df_landmark=df_landmark, model=model_x, axis="x", use_dt=use_dt,
        sid_col=sid_col, age_col=age_col, x_col=x_col, y_col="y",
        title=title, out_html=out_html, tick_mm=tick_mm
    )

def plot_all_subjects_landmark_nexty(
    df_landmark, model_y, use_dt=True,
    sid_col="sid", age_col="age", y_col="y",
    title="", out_html=None, tick_mm=4.0
):
    # forward x_col just to satisfy the generic signature; it won't be used
    return plot_all_subjects_landmark_nextaxis(
        df_landmark=df_landmark, model=model_y, axis="y", use_dt=use_dt,
        sid_col=sid_col, age_col=age_col, x_col="x", y_col=y_col,
        title=title, out_html=out_html, tick_mm=tick_mm
    )
def _group_next_rows(df, use_dt=True, age_col="age", x_col="x", y_col="y", sid_col="sid"):
    """
    Build next-step rows per sid for both x and y.
    Columns: sid, age_t, age_tp1, x_t, y_t, dt, x_true, y_true
    """
    rows = []
    for sid, g in df.groupby(sid_col):
        g = g[[age_col, x_col, y_col]].dropna().sort_values(age_col)
        if len(g) < 2:
            continue
        age = g[age_col].to_numpy(float)
        x   = g[x_col].to_numpy(float)
        y   = g[y_col].to_numpy(float)
        dt  = np.diff(age) if use_dt else np.full(len(age)-1, np.nan, float)
        for i in range(len(age) - 1):
            rows.append({
                "sid": sid,
                "age_t":   age[i],
                "age_tp1": age[i+1],
                "x_t":     x[i],
                "y_t":     y[i],
                "dt":      float(dt[i]) if use_dt else None,
                "x_true":  x[i+1],
                "y_true":  y[i+1],
            })
    return pd.DataFrame(rows)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

def plot_all_subjects_landmark_nextxy_circles(
    df_landmark, model_x, model_y, use_dt=True,
    sid_col="sid", age_col="age", x_col="x", y_col="y",
    title="", out_html=None, circle_mm=2.0, center_dot=False
):
    if df_landmark is None or df_landmark.empty:
        return None

    # Build next-step table (x_t,y_t -> x_{t+1},y_{t+1})
    t = _group_next_rows(
        df_landmark, use_dt=use_dt,
        age_col=age_col, x_col=x_col, y_col=y_col, sid_col=sid_col
    )
    if t is None or t.empty:
        return None

    # Predict next-step (vectorized with safe fallback)
    if use_dt and "dt" in t.columns:
        Xx = t[["x_t", "dt"]].to_numpy(float)
        Xy = t[["y_t", "dt"]].to_numpy(float)
    else:
        Xx = t[["x_t"]].to_numpy(float)
        Xy = t[["y_t"]].to_numpy(float)

    try:
        x_pred = np.asarray(model_x.predict(Xx)).reshape(-1)
    except Exception:
        x_pred = np.asarray([predict_next_x(model_x, xt, dti if use_dt else None)
                             for xt, dti in zip(t["x_t"], t.get("dt", [None]*len(t)))])
    try:
        y_pred = np.asarray(model_y.predict(Xy)).reshape(-1)
    except Exception:
        y_pred = np.asarray([predict_next_x(model_y, yt, dti if use_dt else None)
                             for yt, dti in zip(t["y_t"], t.get("dt", [None]*len(t)))])

    t = t.copy()
    t["x_pred"] = x_pred
    t["y_pred"] = y_pred

    fig = go.Figure()

    # ---- Actual XY paths (blue) ----
    first_actual = True
    for sid_val, g in df_landmark.groupby(sid_col):
        g = g[[age_col, x_col, y_col]].dropna().sort_values(age_col)
        if len(g) < 2:
            continue
        # Repeat sid for hover (don't rely on g[sid_col], it's not in 'g' anymore)
        sid_repeat = np.full(len(g), str(sid_val), dtype=object)
        fig.add_trace(go.Scatter(
            x=g[x_col], y=g[y_col],
            mode="lines",
            # line=dict(color="rgba(0,102,255,0.350)", width=2),
            line=dict(width=2, dash="dot", color="black"),
            name="Actual",
            legendgroup="actual",
            showlegend=first_actual,
            customdata=np.column_stack([sid_repeat]),
            hovertemplate="sid=%{customdata[0]}<br>x=%{x:.3f}<br>y=%{y:.3f}<extra></extra>",
        ))
        first_actual = False

    # ---- Predicted next-step per sid (red; semi-transparent connected line) ----
    first_pred = True
    for sid_val, g in t.groupby("sid"):
        custom = np.column_stack([np.full(len(g), str(sid_val), dtype=object), g["age_tp1"].values])
        fig.add_trace(go.Scatter(
            x=g["x_pred"], y=g["y_pred"],
            mode="lines+markers",
            line=dict(color="rgba(220,0,0,1)", width=2),
            marker=dict(color="red", size=6),
            name="Predicted",
            legendgroup="pred",
            showlegend=first_pred,
            customdata=custom,
            hovertemplate=(
                "sid=%{customdata[0]}<br>age(t+1)=%{customdata[1]:.3f}"
                "<br>pred x=%{x:.3f}<br>pred y=%{y:.3f}"
                f"<br>circle radius={circle_mm:.2f} mm<extra></extra>"
            ),
        ))
        first_pred = False

    # ---- Circle rings at each predicted point (shapes don't have hover) ----
    for xp, yp in zip(t["x_pred"], t["y_pred"]):
        fig.add_shape(
            type="circle", xref="x", yref="y",
            x0=xp - circle_mm, x1=xp + circle_mm,
            y0=yp - circle_mm, y1=yp + circle_mm,
            line=dict(width=1, dash="dot", color="red"),
            layer="above"
        )

    # Optional center dot for easier hovering on rings
    if center_dot:
        fig.add_trace(go.Scatter(
            x=t["x_pred"], y=t["y_pred"],
            mode="markers", marker=dict(size=4, color="red"),
            name="Pred center", legendgroup="pred", showlegend=False,
            hoverinfo="skip"
        ))

    # Axes & legend on the right
    all_x = pd.concat([df_landmark[x_col], pd.Series(t["x_pred"])], ignore_index=True)
    all_y = pd.concat([df_landmark[y_col], pd.Series(t["y_pred"])], ignore_index=True)
    xmin, xmax = float(np.nanmin(all_x)), float(np.nanmax(all_x))
    ymin, ymax = float(np.nanmin(all_y)), float(np.nanmax(all_y))
    pad_x = max(circle_mm * 1.2, 0.02 * (xmax - xmin + 1e-9))
    pad_y = max(circle_mm * 1.2, 0.02 * (ymax - ymin + 1e-9))

    fig.update_layout(
        title=title or "Next-step (x, y) predictions — circles",
        xaxis=dict(title="X (mm)", range=[xmin - pad_x, xmax + pad_x]),
        yaxis=dict(title="Y (mm)", range=[ymin - pad_y, ymax + pad_y],
                   scaleanchor="x", scaleratio=1),
        template="plotly_white",
        height=640,
        legend=dict(
            orientation="v",
            x=1.02, xanchor="left",
            y=1.0,  yanchor="top",
            bgcolor="rgba(255,255,255,0.6)",
            bordercolor="lightgray",
            borderwidth=1
        ),
        margin=dict(r=140)
    )

    if out_html:
        Path(out_html).parent.mkdir(parents=True, exist_ok=True)
        fig.write_html(out_html)
    return fig




def make_report_row(group, landmark, axis, train_info, test_metrics, plot_path):
    """
    Build a report row merging train cv stats and test MAE.
    """
    train_info = train_info or {}
    d = {
        "group": group,
        "landmark": landmark,
        "axis": axis,
        "train_cv_MAE_mean": train_info.get("cv_MAE(mean)"),
        "train_cv_MAE_std": train_info.get("cv_MAE(std)"),
        "train_n_pairs": train_info.get("n_pairs"),
        "test_MAE": test_metrics.get(f"test_MAE_{axis}") if test_metrics else np.nan,
        "plot_path": plot_path
    }
    return d


## train

In [15]:
import pickle
from pathlib import Path

def save_model(m, name, meta=None):
    """
    Save a model (and optional metadata) as a .pkl.
    Returns the path string.
    """
    p = Path(name)
    if not p.suffix:
        p = p.with_suffix(".pkl")
    p.parent.mkdir(parents=True, exist_ok=True)

    bundle = {"model": m, "meta": meta}
    with open(p, "wb") as f:
        pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)
    return str(p)

def load_model(name):
    """
    Load a model saved by save_model(...).
    Returns (model, meta_dict_or_None).
    """
    p = Path(name)
    with open(p, "rb") as f:
        bundle = pickle.load(f)
    return bundle["model"], bundle.get("meta")


In [16]:
import os, pandas as pd
os.makedirs("./plots", exist_ok=True)
os.makedirs("./models", exist_ok=True)

report_rows = []
# landmarks = [ 'ans']

for l in landmarks:
    # -------------------- MALE --------------------
    tr_m = train_m[train_m["landmark"] == l]
    te_m = test_m[test_m["landmark"] == l]

    m_x, info_mx, meta_mx = fit_nextx_regressor(tr_m, use_dt=True, alpha=1.0, cv_splits=5)
    m_y, info_my, meta_my = fit_nextx_regressor(tr_m, use_dt=True, alpha=1.0, cv_splits=5, x_col="y")

    metrics_m, _, _ = eval_nextxy_test((m_x, m_y), te_m, l, use_dt=True)

    male_plot_x  = f"./plots/{l}_male_TEST_age_vs_x_ticks.html"
    male_plot_y = f"./plots/{l}_male_TEST_age_vs_y_ticks.html"
    male_plot_xy = f"./plots/{l}_male_TEST_xy_circles.html"

    male_model_x = f"./models/{l}_male_x_.pkl"
    male_model_y = f"./models/{l}_male_y_.pkl"

    # Age→X (ticks)
    save_model(m_x,male_model_x,info_mx)
    save_model(m_y,male_model_y,info_my)
    
    plot_all_subjects_landmark_nextx(
        te_m, model_x=m_x, use_dt=True,
        title=f"{l} | MALE | Next-step X on TEST (MAE={metrics_m['test_MAE_x']:.3f})",
        out_html=male_plot_x, tick_mm=4.0
    )

    plot_all_subjects_landmark_nextx(
        te_m, model_x=m_y, use_dt=True,x_col='y',
        title=f"{l} | MALE | Next-step Y on TEST (MAE={metrics_m['test_MAE_y']:.3f})",
        out_html=male_plot_y, tick_mm=4.0
    )

    # X–Y (circles)
    plot_all_subjects_landmark_nextxy_circles(
        te_m, model_x=m_x, model_y=m_y, use_dt=True,
        title=f"{l} | MALE | X–Y — next-step circles",
        out_html=male_plot_xy, circle_mm=0.0, center_dot=False
    )
    

    report_rows.append(make_report_row("male", l, "x", info_mx, metrics_m, male_plot_x))
    report_rows.append(make_report_row("male", l, "y", info_my, metrics_m, male_plot_xy))  # link XY plot for Y

    # -------------------- FEMALE --------------------
    tr_f = train_f[train_f["landmark"] == l]
    te_f = test_f[test_f["landmark"] == l]

    f_x, info_fx, meta_fx = fit_nextx_regressor(tr_f, use_dt=True, alpha=1.0, cv_splits=5)
    f_y, info_fy, meta_fy = fit_nextx_regressor(tr_f, use_dt=True, alpha=1.0, cv_splits=5, x_col="y")

    metrics_f, _, _ = eval_nextxy_test((f_x, f_y), te_f, l, use_dt=True)

    female_plot_x  = f"./plots/{l}_female_TEST_age_vs_x_ticks.html"
    female_plot_y  = f"./plots/{l}_female_TEST_age_vs_y_ticks.html"
    female_plot_xy = f"./plots/{l}_female_TEST_xy_circles.html"

    female_model_x = f"./models/{l}_female_x_.pkl"
    female_model_y = f"./models/{l}_female_y_.pkl"

    # Age→X (ticks)
    save_model(m_x,female_model_x,info_fx)
    save_model(m_y,female_model_y,info_fy)

    plot_all_subjects_landmark_nextx(
        te_f, model_x=f_x, use_dt=True,
        title=f"{l} | FEMALE | Next-step X on TEST (MAE={metrics_f['test_MAE_x']:.3f})",
        out_html=female_plot_x, tick_mm=4.0
    )
    plot_all_subjects_landmark_nextx(
        te_f, model_x=f_y, use_dt=True,x_col='y',
        title=f"{l} | FEMALE | Next-step X on TEST (MAE={metrics_f['test_MAE_y']:.3f})",
        out_html=female_plot_y, tick_mm=4.0
    )

    plot_all_subjects_landmark_nextxy_circles(
        te_f, model_x=f_x, model_y=f_y, use_dt=True,
        title=f"{l} | FEMALE | X–Y — next-step circles",
        out_html=female_plot_xy, circle_mm=0.0, center_dot=False
    )

    report_rows.append(make_report_row("female", l, "x", info_fx, metrics_f, female_plot_x))
    report_rows.append(make_report_row("female", l, "y", info_fy, metrics_f, female_plot_xy))

    # -------------------- BOTH (M+F) --------------------
    tr_b = train_all[train_all["landmark"] == l]
    te_b = test_all[test_all["landmark"] == l]

    b_x, info_bx, meta_bx = fit_nextx_regressor(tr_b, use_dt=True, alpha=1.0, cv_splits=5)
    b_y, info_by, meta_by = fit_nextx_regressor(tr_b, use_dt=True, alpha=1.0, cv_splits=5, x_col="y")

    metrics_b, _, _ = eval_nextxy_test((b_x, b_y), te_b, l, use_dt=True)

    both_plot_x  = f"./plots/{l}_both_TEST_age_vs_x_ticks.html"
    both_plot_y  = f"./plots/{l}_both_TEST_age_vs_y_ticks.html"
    both_plot_xy = f"./plots/{l}_both_TEST_xy_circles.html"

    both_model_x = f"./models/{l}_both_x_.pkl"
    both_model_y = f"./models/{l}_both_y_.pkl"

    # Age→X (ticks)
    save_model(m_x,both_model_x,info_bx)
    save_model(m_y,both_model_y,info_by)

    plot_all_subjects_landmark_nextx(
        te_b, model_x=b_x, use_dt=True,
        title=f"{l} | BOTH | Next-step X on TEST (MAE={metrics_b['test_MAE_x']:.3f})",
        out_html=both_plot_x, tick_mm=4.0
    )
    plot_all_subjects_landmark_nextx(
        te_b, model_x=b_y, use_dt=True,x_col='y',
        title=f"{l} | BOTH | Next-step y on TEST (MAE={metrics_b['test_MAE_y']:.3f})",
        out_html=both_plot_y, tick_mm=4.0
    )
    plot_all_subjects_landmark_nextxy_circles(
        te_b, model_x=b_x, model_y=b_y, use_dt=True,
        title=f"{l} | BOTH | X–Y — next-step circles",
        out_html=both_plot_xy, circle_mm=0.0, center_dot=False
    )

    report_rows.append(make_report_row("both", l, "x", info_bx, metrics_b, both_plot_x))
    report_rows.append(make_report_row("both", l, "y", info_by, metrics_b, both_plot_xy))

# Final report
report_df = pd.DataFrame(report_rows)
report_df.to_csv("./nextxy_report_TEST.csv", index=False)
print("Done. Report rows:", len(report_df))


Done. Report rows: 156
